In [1]:
import itertools
import math
from pathlib import Path

import numpy as np
import pandas as pd
import simpy
from sim_tools.distributions import GroupedContinuousEmpirical

In [9]:
class Exponential:
    def __init__(self, mean, random_seed=None):
        self.mean = float(mean)
        self.rng = np.random.default_rng(seed=random_seed)

    def sample(self, size=None):
        return self.rng.exponential(self.mean, size=size)


class Lognormal:
    def __init__(self, mean, sd, random_seed=None):
        mean = float(mean)
        sd = float(sd)

        sigma2 = math.log(1.0 + (sd ** 2) / (mean ** 2))
        self.mu = math.log(mean) - sigma2 / 2.0
        self.sigma = math.sqrt(sigma2)

        self.rng = np.random.default_rng(seed=random_seed)

    def sample(self, size=None):
        return self.rng.lognormal(self.mu, self.sigma, size=size)


class HyperExponential2:
    def __init__(self, mean, sd, random_seed=None):
        mean = float(mean)
        sd = float(sd)

        cv2 = (sd / mean) ** 2
        if cv2 <= 1.0:
            raise ValueError("HyperExponential requires sd > mean")

        self.p = 0.5 * (1.0 + math.sqrt((cv2 - 1.0) / (cv2 + 1.0)))
        self.lambda1 = 2.0 * self.p / mean
        self.lambda2 = 2.0 * (1.0 - self.p) / mean

        self.rng = np.random.default_rng(seed=random_seed)

    def sample(self, size=None):
        if size is None:
            if self.rng.random() < self.p:
                return self.rng.exponential(1.0 / self.lambda1)
            return self.rng.exponential(1.0 / self.lambda2)

        u = self.rng.random(size=size)
        x1 = self.rng.exponential(1.0 / self.lambda1, size=size)
        x2 = self.rng.exponential(1.0 / self.lambda2, size=size)
        return np.where(u < self.p, x1, x2)

In [10]:
class GroupedEmpirical:
    def __init__(self, csv_path, random_seed=None):
        self.df = pd.read_csv(csv_path)
        self.rng = np.random.default_rng(seed=random_seed)

        weights = self.df["y"].astype(float).values
        self.prob = weights / weights.sum()

        self.lower = self.df["lower_bound"].astype(float).values
        self.upper = self.df["upper_bound"].astype(float).values

    def sample(self):
        idx = self.rng.choice(len(self.prob), p=self.prob)
        return self.rng.uniform(self.lower[idx], self.upper[idx])

In [11]:
class Scenario:
    def __init__(self, beds=24, random_seed=42, elective_iat_csv="data(in).csv"):
        self.beds = int(beds)
        self.random_seed = int(random_seed)

        arrival_scale = 1.0

        self.arrival_dist = {
            "ae": Exponential(22.72 * arrival_scale, random_seed + 1),
            "ward": Exponential(26.00 * arrival_scale, random_seed + 2),
            "emergency": Exponential(37.00 * arrival_scale, random_seed + 3),
            "other": Exponential(47.20 * arrival_scale, random_seed + 4),
            "xray": Exponential(575.00 * arrival_scale, random_seed + 5),

            # Empirical elective inter-arrival
            "elective": GroupedEmpirical(elective_iat_csv, random_seed + 6),
        }

        self.los_dist = {
            "ae": HyperExponential2(109.0, 218.0, random_seed + 11),
            "ward": HyperExponential2(152.0, 238.0, random_seed + 12),
            "emergency": HyperExponential2(119.0, 187.0, random_seed + 13),
            "other": Lognormal(182.0, 345.0, random_seed + 14),
            "xray": Lognormal(79.0, 93.0, random_seed + 15),
            "elective": Lognormal(31.0, 42.0, random_seed + 16),
        }

        self.cleaning_time = 5.0
        self.warm_up = 30 * 24
        self.run_length = 365 * 24

        self.beds_resource = None
        self.auditor = None

In [12]:
class Auditor:
    def __init__(self, warm_up, run_length):
        self.warm_up = warm_up
        self.run_length = run_length
        self.end_time = warm_up + run_length

        self.metrics = {
            "occupied_bed_area": 0.0,
            "cancellations": 0,
            "admissions": 0,
        }

        self.last_time = 0.0
        self.last_occupied = 0

    def update_occupied(self, now, occupied):
        start = max(self.last_time, self.warm_up)
        stop = min(now, self.end_time)

        if stop > start:
            self.metrics["occupied_bed_area"] += self.last_occupied * (stop - start)

        self.last_time = now
        self.last_occupied = occupied

    def record_cancellation(self, now):
        if now >= self.warm_up:
            self.metrics["cancellations"] += 1

    def record_admission(self, now):
        if now >= self.warm_up:
            self.metrics["admissions"] += 1

    def finalize(self):
        self.update_occupied(self.end_time, self.last_occupied)

    def summary(self, beds):
        mean_occ = self.metrics["occupied_bed_area"] / self.run_length
        return {
            "mean_occupied": mean_occ,
            "occupancy_pct": 100 * mean_occ / beds,
            "cancellations": self.metrics["cancellations"],
            "admissions": self.metrics["admissions"],
        }

In [13]:
class Patient:
    def __init__(self, source, env, args):
        self.source = source
        self.env = env
        self.args = args

    def service(self):
        beds = self.args.beds_resource
        auditor = self.args.auditor

        if self.source == "elective":
            if beds.level < 1:
                auditor.record_cancellation(self.env.now)
                return

        yield beds.get(1)
        auditor.update_occupied(self.env.now, self.args.beds - beds.level)
        auditor.record_admission(self.env.now)

        los = self.args.los_dist[self.source].sample()
        yield self.env.timeout(los)

        auditor.update_occupied(self.env.now, self.args.beds - beds.level - 1)

        yield self.env.timeout(self.args.cleaning_time)
        yield beds.put(1)

In [14]:
class CCUBedModel:
    def __init__(self, env, args):
        self.env = env
        self.args = args

    def arrivals(self, source):
        while True:
            iat = self.args.arrival_dist[source].sample()
            yield self.env.timeout(iat)

            patient = Patient(source, self.env, self.args)
            self.env.process(patient.service())

In [18]:
# ============================================================
# RUN EMPIRICAL MODEL FOR BEDS 22–29
# ============================================================

def run_empirical_scenarios(bed_range=range(22, 30), n_reps=20, csv="data.csv"):
    results = []

    for beds in bed_range:
        rep_results = []

        for i in range(n_reps):
            res = single_run(beds=beds, seed=1000 + i, csv=csv)
            rep_results.append(res)

        df = pd.DataFrame(rep_results)

        # Average across replications
        summary = {
            "beds": beds,
            "mean_occupied": df["mean_occupied"].mean(),
            "occupancy_pct": df["occupancy_pct"].mean(),
            "cancellations": df["cancellations"].mean(),
            "admissions": df["admissions"].mean(),
        }

        results.append(summary)

    return pd.DataFrame(results)

In [20]:
empirical_results = run_empirical_scenarios(
    bed_range=range(22, 30),
    n_reps=20,
    csv="data(in).csv"
)

print(empirical_results.round(2).to_string(index=False))

 beds  mean_occupied  occupancy_pct  cancellations  admissions
   22          18.79          85.39         155.85     1412.30
   23          18.99          82.55         115.75     1452.65
   24          19.14          79.75          85.35     1483.25
   25          19.26          77.04          61.10     1507.65
   26          19.34          74.39          43.85     1524.90
   27          19.42          71.91          29.55     1539.35
   28          19.47          69.52          19.45     1549.50
   29          19.50          67.25          11.70     1557.25


#### Description

Using the empirical inter-arrival distribution resulted in consistently higher cancellation rates compared to the published results. While occupancy trends remained similar, cancellations did not approach zero at higher bed capacities. This difference is attributed to the empirical distribution containing short inter-arrival times, which create bursts of elective admissions. These bursts increase short-term demand peaks, leading to higher cancellation probabilities. This highlights the importance of arrival variability and scheduling structure in determining system performance